In [1]:
!pip install azure-ai-ml azure-identity scikit-learn pandas --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adlfs 2025.8.0 requires fsspec>=2023.12.0, but you have fsspec 2023.10.0 which is incompatible.
azure-cli 2.81.0 requires azure-datalake-store~=1.0.1, but you have azure-datalake-store 0.0.53 which is incompatible.
azure-cli 2.81.0 requires azure-keyvault-keys==4.11.0, but you have azure-keyvault-keys 4.8.0 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-keyvault==12.1.0, but you have azure-mgmt-keyvault 10.3.1 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-storage==24.0.0, but you have azure-mgmt-storage 22.0.0 which is incompatible.
azure-cli 2.81.0 requires websocket-client~=1.3.1, but you have websocket-client 1.9.0 which is incompatible.
azureml-automl-dnn-nlp 1.61.0 requires torch==2.2.2, but you have torch 2.9.1 which is incompatible.
azureml-automl-runtime 1.61.0 requires psutil<5.

In [2]:
import pandas as pd

data = {
    "hours_study": [2, 6, 0, 4, 1, 3, 6, 5, 4, 5, 4, 6, 8, 0, 6, 5],
    "attendance":  [60, 40, 13, 65, 45, 24, 70, 50, 35, 75, 55, 46, 80, 65, 70, 60],
    "marks":       [40, 80, 20, 75, 35, 50, 85, 65, 45, 90, 60, 72, 95, 25, 82, 70]
}

df = pd.DataFrame(data)
df.to_csv("students.csv", index=False)
print(df)

    hours_study  attendance  marks
0             2          60     40
1             6          40     80
2             0          13     20
3             4          65     75
4             1          45     35
5             3          24     50
6             6          70     85
7             5          50     65
8             4          35     45
9             5          75     90
10            4          55     60
11            6          46     72
12            8          80     95
13            0          65     25
14            6          70     82
15            5          60     70


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import joblib, os

X = df[["hours_study", "attendance"]]
y = df["marks"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

score = model.score(X_test, y_test)
print(f"R2 Score: {score:.2f}")

os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/model.pkl")
print("Model saved!")

R2 Score: 0.53
Model saved!


In [4]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected to workspace:", ml_client.workspace_name)

Found the config file in: /config.json


Connected to workspace: madhumodel1977


In [5]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

registered_model = ml_client.models.create_or_update(
    Model(
        path="model/",
        name="student-marks-model",
        type=AssetTypes.CUSTOM_MODEL,
        description="Predicts student marks from study hours and attendance"
    )
)
print("Model registered! Version:", registered_model.version)

Uploading model (0.14 MBs): 100%|██████████| 137905/137905 [00:00<00:00, 12082038.12it/s]




Model registered! Version: 1


In [24]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

ENDPOINT_NAME = "student-endpoint-v2"   # new name to avoid conflicts

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Student marks predictor",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

Endpoint created: student-endpoint-v2


In [29]:
ml_client.online_deployments.begin_delete(
    name="blue",
    endpoint_name=ENDPOINT_NAME
).result()
print("Old deployment deleted!")

Old deployment deleted!


In [30]:
from azure.ai.ml.entities import (
    ManagedOnlineDeployment, CodeConfiguration, Environment
)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=ml_client.models.get("student-marks-model", label="latest"),
    code_configuration=CodeConfiguration(
        code="src",
        scoring_script="score.py"
    ),
    environment=Environment(
        conda_file="conda.yml",
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    ),
    instance_type="Standard_F2s_v2",
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment done!")

endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic routed!")

Check: endpoint student-endpoint-v2 exists
Uploading src (0.0 MBs): 100%|██████████| 958/958 [00:00<00:00, 29738.47it/s]


Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


......................................................................................Deployment done!
Traffic routed!


Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-20T17:07:40.401548Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-20T17:07:40.684692Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-20T17:08:31.791695Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-20T17:08:31.791695Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-20T17:08:31.860459Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-20T17:08:31.926104Z, Message: Started container inference-server

Container logs:
  File "/var/azureml-app/src/score.py", line 11, in init
    model = joblib.load(model_path)
  File "/azureml-envs/azureml_75

In [31]:
import urllib.request, json

endpoint_info = ml_client.online_endpoints.get(ENDPOINT_NAME)
scoring_url = endpoint_info.scoring_uri
api_key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key

print("Public URL:", scoring_url)

payload = json.dumps({"data": [[7, 85]]}).encode()

req = urllib.request.Request(
    scoring_url,
    data=payload,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
)

with urllib.request.urlopen(req) as resp:
    print("Prediction:", json.loads(resp.read()))

Public URL: https://student-endpoint-v2.centralindia.inference.ml.azure.com/score
Prediction: {"predicted_marks": [89.9]}
